# 第 02 章　怎样把不同分辨率、不同坐标系的数据放到同一张网格？

> **本章问题**：一张 26.7 m 的 UTM 高程、一张 250 m 的 NDVI、一张 10 km 的土壤水分，怎样落到 LST 的同一张 176×106 网格上，而且知道对齐有没有做对？

## 地理情境

甘南县的六张栅格覆盖同一片土地，却各自活在不同的空间参照里。DEM 用 UTM 52N 的米制坐标记录地形，NDVI 是约 250 m 的经纬度格网，降水和土壤水分则是几公里到十公里的粗格网。LST 的 176×106 网格像一张“共同语言”的底纸：只有把五个预测变量都重采样到这张底纸上，才能逐像元比较、建模和制图。

对齐不是“把图缩小”这么简单。它同时改变像元大小、原点、投影和缺测定义；每一次重采样都在回答一个地理问题：新的 1 km 格子，应该由源数据的哪些像元、以什么权重决定？

## 学完能做什么

- 读出参考网格的 `transform`、尺寸、CRS、NoData，并把它写成可复用的检查清单；
- 用 `rasterio.warp.reproject` 把不同 CRS、不同分辨率的数据对齐到同一网格；
- 解释双线性与最近邻的差别，并为每个变量选择合适的方法；
- 用断言和审计表核对“仿射变换、尺寸、CRS、有效/缺测像元数”，而不是只看图“差不多”；
- 把对齐结果写成一个单波段 GeoTIFF，再读回来核对，形成可交付的数据产品。

## 2.1　先认参考网格

先选一张“底纸”，其他栅格都要对齐到它。本教程用 `GNLST.tif`。它的 `transform` 记录像元位置和大小，`shape` 记录行列数，`crs` 说明坐标怎么解释，`nodata` 则标出不能参与计算的值。下面的练习都会拿这四项作基准。


In [ ]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import rasterio
import geopandas as gpd
from rasterio.features import geometry_mask
from rasterio.warp import transform_bounds
from scipy.ndimage import distance_transform_edt

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_style as gs

gs.setup()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 48)

T0 = time.time()
RUN_START = datetime.now().astimezone()
print("Python:", sys.executable)
print("Conda:", os.environ.get("CONDA_PREFIX"))
print("rasterio:", rasterio.__version__, "| geopandas:", gpd.__version__)
print("项目根目录:", PROJECT)
print("运行开始:", RUN_START.isoformat(timespec="seconds"))

## 2.2　把参考网格的四件事读出来

请运行下面的单元，逐项读出 `GNLST.tif` 的 `transform`、尺寸、CRS、NoData 和空间范围。我们把结果写入 `02_reference_grid.csv`，之后每个输入栅格都可以拿它来核对。


In [ ]:
with rasterio.open(gd.RAW / gd.TARGET) as ds:
    REF = {
        "transform": ds.transform,
        "crs": ds.crs,
        "crs_text": ds.crs.to_string(),
        "height": ds.height,
        "width": ds.width,
        "nodata": ds.nodata,
        "bounds": tuple(ds.bounds),
        "res": ds.res,
        "extent": gd.extent_of(ds.transform, (ds.height, ds.width)),
        "valid_count": int((ds.read(1) != ds.nodata).sum()),
    }

t = REF["transform"]
print("参考网格：GNLST.tif")
print(f"  transform = Affine({t.a:.12f}, {t.b}, {t.c:.12f},")
print(f"                   {t.d}, {t.e:.12f}, {t.f:.12f})")
print(f"  尺寸       = {REF['width']} 列 × {REF['height']} 行")
print(f"  CRS        = {REF['crs_text']}")
print(f"  NoData     = {REF['nodata']:.6g}")
print(f"  像元大小   = {REF['res'][0]:.12f}° × {REF['res'][1]:.12f}°")
print(f"  范围       = {REF['bounds']}")
print(f"  有效像元   = {REF['valid_count']:,} / {REF['height'] * REF['width']:,}")

ref_table = pd.DataFrame([
    ("文件", "GNLST.tif"),
    ("transform", f"({t.a:.12f}, {t.b}, {t.c:.12f}, {t.d}, {t.e:.12f}, {t.f:.12f})"),
    ("宽度（列）", REF["width"]),
    ("高度（行）", REF["height"]),
    ("CRS", REF["crs_text"]),
    ("NoData", f"{REF['nodata']:.6g}"),
    ("像元大小（度）", f"{REF['res'][0]:.12f} × {REF['res'][1]:.12f}"),
    ("范围（度）", f"{REF['bounds'][0]:.6f}, {REF['bounds'][1]:.6f}, "
                  f"{REF['bounds'][2]:.6f}, {REF['bounds'][3]:.6f}"),
    ("有效像元", REF["valid_count"]),
], columns=["项目", "值"])
ref_table.to_csv(gd.TABLES / "02_reference_grid.csv", index=False, encoding="utf-8-sig")
print("\n已保存：outputs/tables/02_reference_grid.csv")

参考网格是 **176 列 × 106 行、EPSG:4326、0.008983°、NoData = -3.4028235e38**，有效像元 **7156** 个。注意这个 `transform` 的 `e` 是负数：图像从上往下存储，北边缘在 `f = 48.535975`，每往下一行纬度减少 0.008983°。

`0.008983°` 常被口语化成“1 km”，但在 48°N，东西向一格约 0.67 km，南北向约 1.0 km。这不是本章的枝节，而是重采样时必须记住的几何背景：目标像元并不是正方形公里格。

## 2.3　再看看五张输入各自的网格

底纸选好了，先别急着重采样。我们先把五个预测变量的原生网格排在一起，重点看像元大小、CRS 和覆盖范围。哪张投影不同？哪几张比目标图幅小？答案都会影响边缘像元能不能保留下来。


In [ ]:
source_rows = []
with rasterio.open(gd.RAW / gd.TARGET) as ds_t:
    target_bounds = tuple(ds_t.bounds)

for key, fname in gd.FEATURE_FILES.items():
    with rasterio.open(gd.RAW / fname) as ds:
        b4326 = transform_bounds(ds.crs, "EPSG:4326", *ds.bounds, densify_pts=21)
        covers = (b4326[0] <= target_bounds[0] and b4326[1] <= target_bounds[1] and
                  b4326[2] >= target_bounds[2] and b4326[3] >= target_bounds[3])
        lat_c = (b4326[1] + b4326[3]) / 2
        if ds.crs.to_epsg() == 4326:
            dx_km = abs(ds.transform.a) * 111.32 * np.cos(np.deg2rad(lat_c))
            dy_km = abs(ds.transform.e) * 110.57
            ground = f"{dx_km:.2f} × {dy_km:.2f} km"
            res_text = f"{abs(ds.transform.a):.6f}°"
        else:
            ground = f"{abs(ds.transform.a) / 1000:.3f} × {abs(ds.transform.e) / 1000:.3f} km"
            res_text = f"{abs(ds.transform.a):.2f} m"
        source_rows.append({
            "变量": gd.FEATURE_LABELS[key],
            "文件": fname,
            "宽度": ds.width,
            "高度": ds.height,
            "像元大小": res_text,
            "地面尺寸（东西×南北）": ground,
            "CRS": ds.crs.to_string(),
            "NoData": "未声明" if ds.nodata is None else f"{ds.nodata:.4g}",
            "覆盖目标范围": "是" if covers else "否（边缘可能缺测）",
            "重采样": "双线性" if key in ("dem", "ndvi", "tem") else "最近邻",
        })

sources = pd.DataFrame(source_rows)
sources.to_csv(gd.TABLES / "02_source_grids.csv", index=False, encoding="utf-8-sig")
print(sources.to_string(index=False))

这张表已经把问题说清楚了：五个变量没有一个与目标网格完全一致。DEM 甚至不在经纬度坐标系里；NDVI 和 TEM 的原生范围比目标范围略小；降水和土壤水分的像元比目标像元大 5–15 倍。**对齐不是可选项，而是建模的前置条件。**

选择重采样方法的原则也在表里：DEM、NDVI、TEM 是连续场，适合双线性；降水和土壤水分是粗网格的聚合量，适合最近邻。理由在 2.6 节展开。

## 2.4　在内存中试对齐

项目里已经有一份 `aligned_stack.npz`。为了让这节练习可重复，我们不覆盖它，而是调用 `gd.build_stack(save=False)`，在内存里重新算一次。代码还会用 SHA-256 和修改时间确认已有文件没有变化。

请看下面几条断言。对齐后的 `transform`、`shape` 和 `crs` 必须与参考网格一致。只要其中一项对不上，就先停下来排查，不要继续建模。


In [ ]:
npz_path = gd.PROCESSED / "aligned_stack.npz"
hash_before = hashlib.sha256(npz_path.read_bytes()).hexdigest()
mtime_before = npz_path.stat().st_mtime

t_align = time.time()
bundle = gd.build_stack(save=False)
align_sec = time.time() - t_align

hash_after = hashlib.sha256(npz_path.read_bytes()).hexdigest()
mtime_after = npz_path.stat().st_mtime
npz_unchanged = (hash_before == hash_after) and (mtime_before == mtime_after)

print(f"build_stack(save=False) 耗时：{align_sec:.2f} s")
print(f"aligned_stack.npz SHA-256（前 16 位）：{hash_before[:16]}")
print(f"核心 npz 未被改写：{npz_unchanged}")

def check(name, ok, detail=""):
    print(f"[assert] {name}: {'PASS' if ok else 'FAIL'}  {detail}")
    assert ok, name

check("尺寸一致", bundle["shape"] == (REF["height"], REF["width"]),
      f"{bundle['shape']} vs {(REF['height'], REF['width'])}")
check("仿射变换一致", np.allclose(np.array(bundle["transform"][:6]),
                                  np.array(tuple(REF["transform"])[:6]), rtol=0, atol=0),
      "6 个参数逐项相等")
check("CRS 一致", bundle["crs"] == REF["crs_text"], bundle["crs"])
check("范围一致", np.allclose(np.array(bundle["extent"]), np.array(REF["extent"]), rtol=0, atol=0),
      f"{bundle['extent']}")
check("目标有效像元数一致", int(bundle["valid"].sum()) == REF["valid_count"],
      f"{int(bundle['valid'].sum())}")
check("核心 npz 未改写", npz_unchanged, "SHA-256 与 mtime 均未变化")

断言全部通过：五个变量都落在 **176×106、EPSG:4326** 的目标网格上，仿射变换六个参数逐项相等，`aligned_stack.npz` 的哈希和修改时间都没有变化。`build_stack` 在本机耗时不到 1 秒，因为目标网格只有 18656 个像元，真正的计算量在源栅格的读取与重投影。

需要说明：这里“一致”的是网格定义，不是数值。`transform` 和 `crs` 对上，只说明每个像元的位置对上了；数值是否合理，还要看下一节的审计表和图。

## 2.5　检查对齐后少了哪些像元

网格对齐了，任务还没结束。重采样可能会让边缘像元变成缺测，所以我们还要数一下有效和缺测像元，并看看它们离县界有多远。如果缺测只贴着边界，常见原因是重采样边缘效应；如果县内出现成片空洞，就该回头检查源数据。


In [ ]:
valid = bundle["valid"]
boundary = gpd.read_file(gd.RAW / "GanNan.shp")
with rasterio.open(gd.RAW / gd.TARGET) as ds_t:
    inside = ~geometry_mask(boundary.geometry, out_shape=(REF["height"], REF["width"]),
                            transform=ds_t.transform, invert=False)
dist_to_edge = distance_transform_edt(inside)

audit_rows = []
for key, fname in gd.FEATURE_FILES.items():
    a = bundle[key]
    finite = np.isfinite(a)
    miss = valid & ~finite
    vals = a[valid & finite]
    if miss.any():
        edge = dist_to_edge[miss]
        where = f"距县界 ≤{edge.max():.1f} 像元（中位数 {np.median(edge):.0f}）"
    else:
        where = "无缺测"
    audit_rows.append({
        "变量": gd.FEATURE_LABELS[key],
        "输入": fname,
        "重采样": "双线性" if key in ("dem", "ndvi", "tem") else "最近邻",
        "目标区内有效": int((valid & finite).sum()),
        "目标区内缺测": int(miss.sum()),
        "缺测率": round(float(miss.sum() / valid.sum()), 4),
        "缺测位置": where,
        "区内最小值": round(float(vals.min()), 4),
        "区内中位数": round(float(np.median(vals)), 4),
        "区内最大值": round(float(vals.max()), 4),
        "区内唯一值数": int(np.unique(vals).size),
    })

audit = pd.DataFrame(audit_rows)
common = valid.copy()
for key in gd.FEATURE_FILES:
    common &= np.isfinite(bundle[key])
lost = valid & ~common
audit.loc[len(audit)] = {
    "变量": "五特征共同有效",
    "输入": "—",
    "重采样": "—",
    "目标区内有效": int(common.sum()),
    "目标区内缺测": int(lost.sum()),
    "缺测率": round(float(lost.sum() / valid.sum()), 4),
    "缺测位置": "目标有效区内的交集",
    "区内最小值": np.nan,
    "区内中位数": np.nan,
    "区内最大值": np.nan,
    "区内唯一值数": np.nan,
}
audit["区内唯一值数"] = pd.array(audit["区内唯一值数"], dtype="Int64")
audit.to_csv(gd.TABLES / "02_alignment_audit.csv", index=False, encoding="utf-8-sig")
print(audit.to_string(index=False))

dem_miss = valid & ~np.isfinite(bundle["dem"])
tem_miss = valid & ~np.isfinite(bundle["tem"])
both_miss = dem_miss & tem_miss
print(f"\n目标有效像元：{int(valid.sum()):,}")
print(f"DEM 缺测：{int(dem_miss.sum())}｜TEM 缺测：{int(tem_miss.sum())}｜两者同时缺测：{int(both_miss.sum())}")
print(f"共同有效：{int(common.sum()):,}（{common.sum() / valid.sum():.1%}）｜损失：{int(lost.sum())}（{lost.sum() / valid.sum():.1%}）")
print(f"DEM 缺测距县界最大距离：{dist_to_edge[dem_miss].max():.2f} 像元")
print(f"TEM 缺测距县界最大距离：{dist_to_edge[tem_miss].max():.2f} 像元")
print(f"NDVI 目标区内整值 0 的个数：{int(np.isclose(bundle['ndvi'][valid], 0).sum())}")

common_table = pd.DataFrame([
    ("目标有效像元", int(valid.sum()), 1.0),
    ("DEM 缺测", int(dem_miss.sum()), dem_miss.sum() / valid.sum()),
    ("TEM 缺测", int(tem_miss.sum()), tem_miss.sum() / valid.sum()),
    ("DEM 与 TEM 同时缺测", int(both_miss.sum()), both_miss.sum() / valid.sum()),
    ("五特征共同有效", int(common.sum()), common.sum() / valid.sum()),
    ("因缺测损失", int(lost.sum()), lost.sum() / valid.sum()),
], columns=["项目", "像元数", "占目标有效区比例"])
common_table["占目标有效区比例"] = common_table["占目标有效区比例"].round(4)
common_table.to_csv(gd.TABLES / "02_common_valid.csv", index=False, encoding="utf-8-sig")

真实结果是：

- 目标有效区 **7156** 个像元；
- DEM 对齐后有效 **7115**、缺 **41**；TEM 有效 **7059**、缺 **97**；
- NDVI、降水、土壤水分在目标区内都完整（缺测 0）；
- 五个特征共同有效 **7051** 个，占目标有效区的 **98.5%**，损失 **105** 个（DEM 与 TEM 有 33 个像元同时缺测，所以不是 41+97=138）。

更关键的是缺测的位置：DEM 的 41 个缺测全部在距县界 **1 个像元**以内，TEM 的 97 个也都在 **1.4 个像元**以内，县内部没有空洞。这说明缺测不是“数据坏了”，而是双线性重采样在源数据掩膜边缘找不到完整的 2×2 邻域：目标像元跨在“有值/无值”的分界上，重采样只能返回 NoData。这是典型的**边缘效应**。

NDVI 在目标区内没有整值 0，最小值为 -0.048。这再次说明 0 是县界外的填充，不是县内的真实观测；对齐过程没有把 0 带进目标区，也不应该把 0 全局改成缺测。

## 2.6　双线性还是最近邻？

重采样方法要跟数据含义走，不是图看起来越平滑越好。

- **DEM、NDVI、TEM** 是连续变量，可以用双线性插值，让邻近源像元按距离加权。
- **降水和土壤水分**来自较粗的聚合网格，本教程用最近邻保留源像元值。这样结果会有块状边界，但不会假装粗网格真的提供了细尺度变化。
- **NDVI 的边界填充值要先核查**。如果 0 被当成有效值参与插值，县界附近就可能被拉低。本例已确认县内没有整值 0。

记住，重采样能统一网格，不能给粗分辨率数据凭空增加细节。


## 2.7　对齐结果预览

下面四幅图都在目标网格上，浅灰是缺测。DEM 的细纹理被聚合成 1 km 左右的块面，NDVI 仍保留较细的空间变化，降水与土壤水分则呈现明显的粗网格块状。把它们放在同一张底图上，才能直观看出“同一地点、不同支撑尺度”的差别。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.8, 10.4))
fig.subplots_adjust(left=0.06, right=0.965, top=0.90, bottom=0.07,
                    hspace=0.30, wspace=0.18)
extent = REF["extent"]

panels = [
    ("dem", "(a) DEM｜双线性", "terrain", 123, 328, "高程（原始像元值）"),
    ("ndvi", "(b) NDVI｜双线性", "YlGn", -0.05, 0.48, "NDVI（原始像元值）"),
    ("precip", "(c) 年降水｜最近邻", "YlGnBu", 592, 736, "降水（原始像元值）"),
    ("soil", "(d) 土壤水分｜最近邻", "BrBG", 11.7, 17.0, "土壤水分（原始像元值）"),
]
for ax, (key, title, cmap, vmin, vmax, cbar) in zip(axes.ravel(), panels):
    miss_n = int((valid & ~np.isfinite(bundle[key])).sum())
    plot_arr = np.where(valid, bundle[key], np.nan)  # 只显示 LST 有效区，县界外置灰
    gs.plot_raster(
        ax, plot_arr, extent, cmap, vmin, vmax,
        title=f"{title}｜缺测 {miss_n}", cbar_label=cbar,
        boundary=boundary, crs=REF["crs"], add_bar=True,
    )

fig.suptitle("图 2-1　五个预测变量对齐到 GNLST 176×106 网格（浅灰为缺测）",
             fontsize=15.0, y=0.965)
fig.text(0.5, 0.012,
         "目标网格：EPSG:4326，0.008983°；DEM/NDVI/TEM 双线性，降水/土壤水分最近邻。"
         "仅显示 LST 有效区（县界内），其余置灰；边界线来自 GanNan.shp；色标单位均为原始像元值。",
         ha="center", va="bottom", fontsize=9.2, color=gs.INK)
gs.save(fig, "02_alignment_demo.png")
plt.show()
plt.close(fig)
print("已保存：outputs/figures/02_alignment_demo.png")

四幅图共享同一套经纬度坐标和同一条县界，可以直接比较：

- DEM 在西部和西北部较高、东部和东南部较低，双线性保留了地形的连续起伏，但 41 个边缘格缺失；
- NDVI 的高值集中在东部和南部，县界内没有整值 0，颜色没有出现“被 0 拉黑”的边缘；
- 年降水呈明显的大块状，这是最近邻保留源像元边界的结果，不是降水真的在 1 km 尺度上突变；
- 土壤水分的块更大，且与降水、NDVI 的空间格局并不完全一致。粗网格变量在 1 km 网格上被“放大”后，看起来像有细节，实际上每个块内部是同一个值。

这就是重采样的诚实边界：它能把不同数据放到同一张底图上，但不能凭空增加源数据没有的空间信息。

## 2.8　目标有效区和共同有效区

建模时，不仅要问“LST 有多少有效像元”，还要问“五个输入是不是在同一个像元上都有值”。看下面的图，橙色标出因为 DEM 或 TEM 边缘缺测而损失的位置。


In [ ]:
valid_arr = np.where(valid, 1.0, np.nan)
common_arr = np.full((REF["height"], REF["width"]), np.nan)
common_arr[common] = 1.0
common_arr[lost] = 2.0

fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.8))
fig.subplots_adjust(left=0.055, right=0.975, top=0.86, bottom=0.12, wspace=0.16)

cmap_a = ListedColormap(["#2a6f97"])
axes[0].set_facecolor(gs.MISSING_COLOR)
axes[0].imshow(np.ma.masked_invalid(valid_arr), extent=extent, origin="upper",
               cmap=cmap_a, interpolation="nearest", aspect="auto", vmin=0, vmax=2, zorder=1)
boundary.to_crs(REF["crs"]).boundary.plot(ax=axes[0], color="#303030", lw=0.8, zorder=5)
gs._format_coord_axes(axes[0], REF["crs"])
axes[0].set_title(f"(a) 目标 LST 有效区：{int(valid.sum()):,} 个像元")
axes[0].text(0.03, 0.96, f"有效 {int(valid.sum()):,}\n缺测 {int((~valid).sum()):,}",
             transform=axes[0].transAxes, ha="left", va="top", fontsize=10,
             color=gs.INK, zorder=20,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.86))
gs.add_north_arrow(axes[0])
gs.add_scalebar(axes[0], extent)

cmap_b = ListedColormap(["#2a6f97", "#d97706"])
norm_b = BoundaryNorm([0.5, 1.5, 2.5], cmap_b.N)
axes[1].set_facecolor(gs.MISSING_COLOR)
axes[1].imshow(np.ma.masked_invalid(common_arr), extent=extent, origin="upper",
               cmap=cmap_b, norm=norm_b, interpolation="nearest", aspect="auto", zorder=1)
boundary.to_crs(REF["crs"]).boundary.plot(ax=axes[1], color="#303030", lw=0.8, zorder=5)
gs._format_coord_axes(axes[1], REF["crs"])
axes[1].set_title(f"(b) 五特征共同有效区：{int(common.sum()):,} 个像元")
axes[1].text(0.03, 0.96,
             f"共同有效 {int(common.sum()):,}（{common.sum() / valid.sum():.1%}）\n"
             f"损失 {int(lost.sum())}（{lost.sum() / valid.sum():.1%}）\n"
             f"DEM 缺 {int(dem_miss.sum())}｜TEM 缺 {int(tem_miss.sum())}｜重叠 {int(both_miss.sum())}",
             transform=axes[1].transAxes, ha="left", va="top", fontsize=9.4,
             color=gs.INK, zorder=20,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.86))
gs.add_north_arrow(axes[1])
gs.add_scalebar(axes[1], extent)
handles = [
    mpatches.Patch(color="#2a6f97", label="共同有效"),
    mpatches.Patch(color="#d97706", label="因 DEM/TEM 边缘缺测损失"),
    mpatches.Patch(color=gs.MISSING_COLOR, label="目标区外/缺测"),
]
axes[1].legend(handles=handles, loc="lower right", fontsize=8.8, ncol=1,
                frameon=True, facecolor="white", edgecolor="none", framealpha=0.92)

fig.suptitle("图 2-2　目标有效区与五特征共同有效区", fontsize=15.0, y=0.965)
gs.save(fig, "02_common_valid.png")
plt.show()
plt.close(fig)
print("已保存：outputs/figures/02_common_valid.png")

共同有效区是 **7051** 个像元，比目标有效区少 **105** 个。损失的像元几乎都贴县界分布，橙色在图上只是沿边界的一圈细线，而不是内部的大块空洞。这个结果对建模有三个直接后果：

1. 训练表会保留 7156 行（含缺测），但完整案例建模时真正可用的是 7051 行；第 3 章会同时保留这两种口径。
2. 缺测不能随便填 0。中位数填补、标准化都必须在训练折内拟合，否则会把测试区信息泄露进训练。
3. 边缘像元同时是“地形梯度最大、混合像元最多”的地方，损失它们会让模型对县界附近的极端地形更不敏感。这是可接受的代价，但要在论文/报告中写明。

## 2.9　写出 GeoTIFF，再读回来检查

最后我们把对齐后的 NDVI 写成一个单波段 GeoTIFF，再重新打开它。请核对驱动、尺寸、CRS、`transform`、NoData 和像元值。这一步能确认输出不只是内存里看着正确，也能被 GIS 软件按正确位置读取。


In [ ]:
demo_path = gd.PREDICTIONS / "02_ndvi_aligned_demo.tif"
profile = dict(
    driver="GTiff", height=REF["height"], width=REF["width"], count=1,
    dtype="float32", crs=REF["crs"], transform=REF["transform"],
    nodata=np.nan, compress="deflate", tiled=True, blockxsize=64, blockysize=64,
)
with rasterio.open(demo_path, "w", **profile) as ds:
    ds.write(bundle["ndvi"].astype("float32"), 1)
    ds.set_band_description(1, "NDVI aligned to GNLST grid")
    ds.update_tags(source="GNNDVI.tif", reference_grid="GNLST.tif",
                   resampling="bilinear")

with rasterio.open(demo_path) as ds:
    back = ds.read(1, masked=True).filled(np.nan).astype("float64")
    print("读回 GeoTIFF：")
    print("  驱动   :", ds.driver)
    print("  尺寸   :", ds.width, "×", ds.height)
    print("  CRS    :", ds.crs)
    print("  transform:", tuple(round(v, 12) for v in tuple(ds.transform)[:6]))
    print("  NoData :", ds.nodata)
    print("  描述   :", ds.descriptions[0])
    print("  标签   :", ds.tags())
    check("写回尺寸一致", (ds.height, ds.width) == (REF["height"], REF["width"]))
    check("写回 CRS 一致", ds.crs.to_string() == REF["crs_text"])
    check("写回 transform 一致",
          np.allclose(np.array(tuple(ds.transform)[:6]),
                      np.array(tuple(REF["transform"])[:6]), rtol=0, atol=0))
    check("写回 NoData 为 NaN", bool(np.isnan(ds.nodata)))
    same_nan = np.array_equal(np.isnan(back), np.isnan(bundle["ndvi"]))
    max_diff = float(np.nanmax(np.abs(back - bundle["ndvi"])))
    check("缺测位置一致", same_nan)
    print(f"  数值最大绝对差：{max_diff:.2e}")

print(f"\n文件大小：{demo_path.stat().st_size / 1024:.1f} KB")
print("已保存：outputs/predictions/02_ndvi_aligned_demo.tif")

读回检查通过了：尺寸、CRS、`transform`、NoData 和缺测位置都一致，数值最大差为 0。现在这张图已经是可以交给 GIS 打开的数据产品。

看起来只是多读了一次文件，却能及时发现坐标写错、CRS 丢失或把 NoData 写成 0 这类问题。


## 地理视角

对齐的本质是空间支撑（spatial support）的转换。DEM 的 26.7 m 像元、NDVI 的 250 m 像元、降水的 5 km 像元，各自代表不同面积上的平均状态；把它们放到 1 km 网格上，不是简单地“变清晰”，而是把不同尺度的过程强行放进同一个分析单元。 geography 里常说的可变面元问题（MAUP）在这里非常具体：同一组变量，换一套格网，相关系数和模型结果都可能变。

甘南县的县界是不规则多边形，目标网格是矩形。边缘像元的缺测不是随机丢失，而是“混合像元”最集中的地方：一个格子可能一半在县内、一半在县外。双线性在掩膜边缘返回 NoData，其实是在提醒我们：这些格子的值本来就不该被当作纯净的县内观测。

## 常见坑

- **只改 `dst_transform`，忘了 `dst_crs`**：轻则位置偏移，重则经纬度被当成米，整张图飞到另一个地方。
- **只设 `src_nodata`，不设 `dst_nodata`**：NoData 可能被当成 0 或极大值参与统计。
- **把 NDVI 的 0 全局改成 NaN**：县界外的填充会被清掉，但县内真实的低值也可能被误删；本数据县内没有 0，仍要先核实再处理。
- **认为最近邻“更保真”**：最近邻保的是原始值，不是空间连续场；对 DEM 用最近邻会产生台阶，对降水用双线性会发明中间值。
- **只看图不看计数**：图上“差不多”，但有效像元可能少了几十个；必须打印断言和审计表。
- **直接覆盖 `aligned_stack.npz`**：本章用 `save=False`，并核对哈希，避免把其他章节依赖的核心产物改掉。
- **把 0.008983° 当正方形公里格**：在 48°N，东西向约 0.67 km，面积约 0.665 km²。

## 练习

把 DEM 和 NDVI 的重采样方法都改成最近邻（其余不变），重新在内存中计算一次对齐，统计目标区内有效像元数和五特征共同有效区，并与本章的双线性结果比较。然后回答：为什么本课最终仍选择双线性？提示：比较缺测数量和空间纹理，不需要重新训练模型。

In [ ]:
runtime_sec = time.time() - T0
summary = {
    "notebook": "notebooks/02_栅格对齐.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "run_started": RUN_START.isoformat(timespec="seconds"),
    "runtime_sec": round(runtime_sec, 1),
    "question": "怎样把不同分辨率、不同坐标系的数据放到同一张网格？",
    "reference_grid": {
        "file": "GNLST.tif",
        "shape": [REF["height"], REF["width"]],
        "crs": REF["crs_text"],
        "res_deg": round(float(REF["res"][0]), 12),
        "nodata": float(REF["nodata"]),
        "valid_pixels": int(REF["valid_count"]),
    },
    "key_numbers": {
        "target_valid": int(valid.sum()),
        "dem_valid": int((valid & np.isfinite(bundle["dem"])).sum()),
        "dem_missing": int(dem_miss.sum()),
        "tem_valid": int((valid & np.isfinite(bundle["tem"])).sum()),
        "tem_missing": int(tem_miss.sum()),
        "ndvi_missing": int((valid & ~np.isfinite(bundle["ndvi"])).sum()),
        "precip_missing": int((valid & ~np.isfinite(bundle["precip"])).sum()),
        "soil_missing": int((valid & ~np.isfinite(bundle["soil"])).sum()),
        "dem_tem_overlap_missing": int(both_miss.sum()),
        "common_valid": int(common.sum()),
        "common_valid_fraction": round(float(common.sum() / valid.sum()), 4),
        "lost_pixels": int(lost.sum()),
        "ndvi_zeros_inside_target": int(np.isclose(bundle["ndvi"][valid], 0).sum()),
        "align_seconds": round(align_sec, 2),
        "npz_sha256_prefix": hash_before[:16],
    },
    "alignment_checks": {
        "shape_equal": True,
        "transform_equal": True,
        "crs_equal": True,
        "extent_equal": True,
        "npz_unchanged": bool(npz_unchanged),
    },
    "figures": [
        "outputs/figures/02_alignment_demo.png",
        "outputs/figures/02_common_valid.png",
    ],
    "tables": [
        "outputs/tables/02_reference_grid.csv",
        "outputs/tables/02_source_grids.csv",
        "outputs/tables/02_alignment_audit.csv",
        "outputs/tables/02_common_valid.csv",
        "outputs/tables/02_run_summary.json",
    ],
    "predictions": [
        "outputs/predictions/02_ndvi_aligned_demo.tif",
    ],
    "unverified": [
        "重采样只解决空间支撑问题，不能修正单位、时间错位或变量间的因果关系。",
        "DEM/TEM 缺测全部位于距县界不超过 1.4 像元的范围内，属于掩膜边缘效应；本课未做额外的边缘填补。",
        "NDVI 的 0 填充仅确认在县界外；其观测时间与来源仍未核实。",
        "降水和土壤水分是 2020 年聚合量，与 LST 的 2021 年不同期。",
    ],
    "notes": "本章用 gd.build_stack(save=False) 在内存重算，未覆盖 data/processed/aligned_stack.npz；只向 outputs/predictions 写了一个小的演示 GeoTIFF。",
}
gd.write_json(summary, "02_run_summary.json")
print(f"运行耗时：{runtime_sec:.1f} s")
print("已保存：outputs/tables/02_run_summary.json")

## 小结

- 五个预测变量都对齐到了 **GNLST 176×106、EPSG:4326** 的网格；`transform`、`shape`、`crs`、`extent` 四项断言全部通过，核心 `aligned_stack.npz` 未被改写。
- DEM 双线性缺 **41**、TEM 双线性缺 **97**，全部在距县界不超过 1.4 像元的范围内；NDVI、降水、土壤水分在目标区内完整。五特征共同有效 **7051**（98.5%），损失 **105** 个像元（其中 33 个是 DEM 与 TEM 同时缺测）。
- NDVI 在目标区内没有整值 0；0 只出现在县界外，不能全局当缺测处理。双线性前若不检查填充值，会把边界外的 0 混进县内。
- 连续场（DEM、NDVI、TEM）用双线性，聚合量（降水、土壤水分）用最近邻；选择依据是变量的物理含义，不是“哪种图更好看”。
- 对齐后的 NDVI 已写成 `outputs/predictions/02_ndvi_aligned_demo.tif` 并读回核对；下一章将用这些对齐结果构建训练表。